# Data preparation and merging

In [1]:
import pandas as pd
raw_application_df = pd.read_csv('raw_data/IS453 Group Assignment - Application Data.csv')
raw_bureau_df = pd.read_csv('raw_data/IS453 Group Assignment - Bureau Data.csv')

In [2]:
raw_bureau_df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.0,NaN,0.0,0,11250.00,11250.0,0.0,0.0,Microloan,-19,NaN
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.0,-2493.0,5476.5,0,38130.84,0.0,0.0,0.0,Consumer credit,-2493,NaN
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.0,-970.0,NaN,0,15570.00,NaN,NaN,0.0,Consumer credit,-967,NaN
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.0,-1513.0,NaN,0,36000.00,0.0,0.0,0.0,Consumer credit,-1508,NaN


In [3]:
#FILTERING IN APPLICATION DATA

#filter out 'FLAG_OWN_REALTY'] == 'Y'
cleaned_application_df = raw_application_df[raw_application_df['FLAG_OWN_REALTY'] != 'Y']

#filter out people who already own a house
allowed_types = ['Rented apartment', 'With parents', 'Municipal apartment', 'Office apartment']
cleaned_application_df = cleaned_application_df[cleaned_application_df['NAME_HOUSING_TYPE'].isin(allowed_types)]

#filter out people under 21
cleaned_application_df = cleaned_application_df[cleaned_application_df['DAYS_BIRTH'] <= -7665]

In [4]:
# CREDIT_CURRENCY:(Assumption: currency 1 is local) -> currency 2,3,4 falls into foreign currency category and we split by counts for those columns
raw_bureau_df['CREDIT_CURRENCY'].value_counts()

CREDIT_CURRENCY
currency 1    1715020
currency 2       1224
currency 3        174
currency 4         10
Name: count, dtype: int64

In [5]:
#AMT_CREDIT_SUM_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,CREDIT_DAY_OVERDUE
import numpy as np

df = raw_bureau_df.copy()

g = df.groupby('SK_ID_CURR')

# max overdue on Active loans; no Active loan -> 0
amt_overdue_client = (
    df.loc[df['CREDIT_ACTIVE'] == 'Active']
    .groupby('SK_ID_CURR')['AMT_CREDIT_SUM_OVERDUE']
    .max()
)

# Map to new column
df['MAX_AMT_CREDIT_SUM_OVERDUE'] = (
    df['SK_ID_CURR']
    .map(amt_overdue_client)
    .fillna(0)
)


# Ratio uses sums per person; same value on every row for that client
sum_credit = g['AMT_CREDIT_SUM'].sum()
sum_debt = g['AMT_CREDIT_SUM_DEBT'].sum()
# total_debt_ratio = sum_credit.div(sum_debt.replace(0, np.nan))
total_debt_ratio = sum_debt.div(sum_credit.replace(0, np.nan))
df['DEBT_RATIO'] = df['SK_ID_CURR'].map(total_debt_ratio)

#taking average overdue days per person
avg_overdue = g['CREDIT_DAY_OVERDUE'].mean()
df['AVERAGE_CREDIT_DAY_OVERDUE'] = df['SK_ID_CURR'].map(avg_overdue)
df[df['SK_ID_CURR']==100002]



,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,MAX_AMT_CREDIT_SUM_OVERDUE,DEBT_RATIO,AVERAGE_CREDIT_DAY_OVERDUE
675684,100002,6158904,Closed,currency 1,-1125,0,-1038.0,-1038.0,NaN,0,40761.000,NaN,NaN,0.0,Credit card,-1038,0.0,0.0,0.284122,0.0
675685,100002,6158905,Closed,currency 1,-476,0,NaN,-48.0,NaN,0,0.000,0.0,NaN,0.0,Credit card,-47,NaN,0.0,0.284122,0.0
675686,100002,6158906,Closed,currency 1,-1437,0,-1072.0,-1185.0,0.000,0,135000.000,0.0,0.000,0.0,Consumer credit,-1185,0.0,0.0,0.284122,0.0
675687,100002,6158907,Closed,currency 1,-1121,0,-911.0,-911.0,3321.000,0,19071.000,NaN,NaN,0.0,Consumer credit,-906,0.0,0.0,0.284122,0.0
675688,100002,6158908,Closed,currency 1,-645,0,85.0,-36.0,5043.645,0,120735.000,0.0,0.000,0.0,Consumer credit,-34,0.0,0.0,0.284122,0.0
675689,100002,6158909,Active,currency 1,-103,0,NaN,NaN,40.500,0,31988.565,0.0,31988.565,0.0,Credit card,-24,0.0,0.0,0.284122,0.0
1337779,100002,6158903,Active,currency 1,-1042,0,780.0,NaN,NaN,0,450000.000,245781.0,0.000,0.0,Consumer credit,-7,0.0,0.0,0.284122,0.0
1486113,100002,6113835,Closed,currency 1,-1043,0,62.0,-967.0,0.000,0,67500.000,NaN,NaN,0.0,Credit card,-758,0.0,0.0,0.284122,0.0


In [6]:
# AMT_CREDIT_SUM_LIMIT
# - TOTAL_CREDIT_LIMIT: Sum up all of the AMT_CREDIT_SUM_LIMIT
df[df['SK_ID_CURR']==100002]
g = df.groupby('SK_ID_CURR')
total_limit = g['AMT_CREDIT_SUM_LIMIT'].sum()
df['TOTAL_CREDIT_LIMIT'] = df['SK_ID_CURR'].map(total_limit)
df[df['SK_ID_CURR']==100002]

# CNT_CREDIT_PROLONG: SUM_CNT_CREDIT_PROLONG
# - sum up how many extensions the guy asked for
g = df.groupby('SK_ID_CURR')
sum_prolong = g['CNT_CREDIT_PROLONG'].sum()
df['SUM_CNT_CREDIT_PROLONG'] = df['SK_ID_CURR'].map(sum_prolong)
df[df['SK_ID_CURR']==100002]

# CREDIT_TYPE to count of TOTAL of the different types of credits.
# Create dummies and attach directly to df
df = pd.concat([df, pd.get_dummies(df['CREDIT_TYPE'], prefix='CREDIT_TYPE')], axis=1)

# Only dummy columns (exclude original CREDIT_TYPE)
credit_type_cols = [col for col in df.columns
                    if col.startswith('CREDIT_TYPE_') and col != 'CREDIT_TYPE']

# Groupby
g = df.groupby('SK_ID_CURR')

# Map counts back
for col in credit_type_cols:
    df[col] = df['SK_ID_CURR'].map(g[col].sum())

df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,CREDIT_TYPE_Interbank credit,CREDIT_TYPE_Loan for business development,CREDIT_TYPE_Loan for purchase of shares (margin lending),CREDIT_TYPE_Loan for the purchase of equipment,CREDIT_TYPE_Loan for working capital replenishment,CREDIT_TYPE_Microloan,CREDIT_TYPE_Mobile operator loan,CREDIT_TYPE_Mortgage,CREDIT_TYPE_Real estate loan,CREDIT_TYPE_Unknown type of loan
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,0,0,0,0,0,0,0,0,0,0
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.0,NaN,0.0,0,...,0,0,0,0,0,2,0,0,0,0
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.0,-2493.0,5476.5,0,...,0,0,0,0,0,0,0,0,0,0
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.0,-970.0,NaN,0,...,0,0,0,0,0,0,0,0,0,0
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.0,-1513.0,NaN,0,...,0,0,0,0,0,12,0,0,0,0


In [7]:

df["LOAN_DURATION_SCHEDULED"] = df["DAYS_CREDIT_ENDDATE"] - df["DAYS_CREDIT"]
df["LOAN_DURATION_ACTUAL"] = df["DAYS_ENDDATE_FACT"] - df["DAYS_CREDIT"]
df["LOAN_DEVIATION"] = df["DAYS_CREDIT_ENDDATE"] - df["DAYS_ENDDATE_FACT"]

# Use only closed loans for these metrics
closed_loans = df[df["CREDIT_ACTIVE"] == "Closed"].copy()
closed_group = closed_loans.groupby("SK_ID_CURR")

# User-level aggregates (closed loans)
max_recency_closed = closed_group["DAYS_ENDDATE_FACT"].max()
avg_duration_actual = closed_group["LOAN_DURATION_ACTUAL"].mean()

# Map DAYS_ENDDATE_FACT features back to bureau rows
df["MAX_RECENCY_CLOSED_LOAN"] = df["SK_ID_CURR"].map(max_recency_closed)
df["AVG_LOAN_DURATION_ACTUAL"] = df["SK_ID_CURR"].map(avg_duration_actual)

# AMT_CREDIT_MAX_OVERDUE features
overdue_group = df.assign(
    AMT_CREDIT_MAX_OVERDUE=df["AMT_CREDIT_MAX_OVERDUE"].fillna(0)
).groupby("SK_ID_CURR")

max_overdue_amount = overdue_group["AMT_CREDIT_MAX_OVERDUE"].max()
count_overdue_loans = overdue_group["AMT_CREDIT_MAX_OVERDUE"].apply(lambda s: (s > 0).sum())

# Map overdue features back to bureau rows
df["MAX_OVERDUE_AMOUNT"] = df["SK_ID_CURR"].map(max_overdue_amount)
df["COUNT_OVERDUE_LOANS"] = df["SK_ID_CURR"].map(count_overdue_loans)
df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,CREDIT_TYPE_Mortgage,CREDIT_TYPE_Real estate loan,CREDIT_TYPE_Unknown type of loan,LOAN_DURATION_SCHEDULED,LOAN_DURATION_ACTUAL,LOAN_DEVIATION,MAX_RECENCY_CLOSED_LOAN,AVG_LOAN_DURATION_ACTUAL,MAX_OVERDUE_AMOUNT,COUNT_OVERDUE_LOANS
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,0,0,0,344.0,344.0,0.0,-153.0,399.600000,77674.500,1
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,0,0,0,1283.0,NaN,NaN,-153.0,399.600000,77674.500,1
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,0,0,0,731.0,NaN,NaN,-153.0,399.600000,77674.500,1
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,0,0,0,NaN,NaN,NaN,-153.0,399.600000,77674.500,1
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,0,0,0,1826.0,NaN,NaN,-153.0,399.600000,77674.500,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.0,NaN,0.0,0,...,0,0,0,14.0,NaN,NaN,NaN,NaN,0.000,0
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.0,-2493.0,5476.5,0,...,0,0,0,215.0,155.0,60.0,-970.0,655.600000,18406.935,5
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.0,-970.0,NaN,0,...,0,0,0,181.0,839.0,-658.0,-970.0,655.600000,18406.935,5
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.0,-1513.0,NaN,0,...,0,0,0,365.0,365.0,0.0,-165.0,219.956522,3759.480,3


In [8]:
# CREDIT_ACTIVE, CREDIT_CURRENCY

g = df.groupby('SK_ID_CURR')

# CREDIT_ACTIVE: count of each status per client
active_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Active').sum())
closed_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Closed').sum())
sold_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Sold').sum())
bad_debt_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Bad debt').sum())

df['TOTAL_ACTIVE'] = df['SK_ID_CURR'].map(active_count)
df['TOTAL_CLOSED'] = df['SK_ID_CURR'].map(closed_count)
df['TOTAL_SOLD'] = df['SK_ID_CURR'].map(sold_count)
df['TOTAL_BAD_DEBT'] = df['SK_ID_CURR'].map(bad_debt_count)

# CREDIT_CURRENCY: local vs foreign loan counts
# Assumption: currency 1 is local, everything else is foreign
local_count = g['CREDIT_CURRENCY'].apply(lambda x: (x == 'currency 1').sum())
foreign_count = g['CREDIT_CURRENCY'].apply(lambda x: (x != 'currency 1').sum())

df['COUNT_LOCAL_CURRENCY_LOANS'] = df['SK_ID_CURR'].map(local_count)
df['COUNT_FOREIGN_CURRENCY_LOANS'] = df['SK_ID_CURR'].map(foreign_count)

# Verify with one customer
df[df['SK_ID_CURR']==215354][['SK_ID_CURR', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
    'TOTAL_ACTIVE', 'TOTAL_CLOSED', 'TOTAL_SOLD', 'TOTAL_BAD_DEBT',
    'COUNT_LOCAL_CURRENCY_LOANS', 'COUNT_FOREIGN_CURRENCY_LOANS']]

,SK_ID_CURR,CREDIT_ACTIVE,CREDIT_CURRENCY,TOTAL_ACTIVE,TOTAL_CLOSED,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS
0,215354,Closed,currency 1,6,5,0,0,11,0
1,215354,Active,currency 1,6,5,0,0,11,0
2,215354,Active,currency 1,6,5,0,0,11,0
3,215354,Active,currency 1,6,5,0,0,11,0
4,215354,Active,currency 1,6,5,0,0,11,0
5,215354,Active,currency 1,6,5,0,0,11,0
6,215354,Active,currency 1,6,5,0,0,11,0
225157,215354,Closed,currency 1,6,5,0,0,11,0
225158,215354,Closed,currency 1,6,5,0,0,11,0
225159,215354,Closed,currency 1,6,5,0,0,11,0


In [9]:

g = df.groupby('SK_ID_CURR')

# DAYS_CREDIT_RECENT
df['DAYS_CREDIT_RECENT'] = df['SK_ID_CURR'].map(g['DAYS_CREDIT'].max())

# DAYS_CREDIT_OLDEST
df['DAYS_CREDIT_OLDEST'] = df['SK_ID_CURR'].map(g['DAYS_CREDIT'].min())

# AVERAGE_CREDIT_DAY_OVERDUE
df['AVERAGE_CREDIT_DAY_OVERDUE'] = df['SK_ID_CURR'].map(g['CREDIT_DAY_OVERDUE'].mean())

# RECENT_LOAN_COUNT
recent_active = df[
    (df['CREDIT_ACTIVE'] == 'Active') &
    (df['DAYS_CREDIT'] >= -100)
]

recent_loan_count = recent_active.groupby('SK_ID_CURR').size()
df['RECENT_LOAN_COUNT'] = df['SK_ID_CURR'].map(recent_loan_count).fillna(0).astype(int)

df[df['SK_ID_CURR'] == 100002]


# AVERAGE_DAYS_OVERDUE
# - take average of DAYS_CREDIT_ENDDATE where:
#   CREDIT_ACTIVE == 'Active'
#   DAYS_CREDIT_ENDDATE < 0
# - if no such rows → 0

# Step 1: filter
active_overdue = df[
    (df['CREDIT_ACTIVE'] == 'Active') &
    (df['DAYS_CREDIT_ENDDATE'] < 0)
]

# Step 2: group and average
avg_days_overdue = active_overdue.groupby('SK_ID_CURR')['DAYS_CREDIT_ENDDATE'].mean()

# Step 3: map back and fill missing with 0
df['AVERAGE_DAYS_OVERDUE'] = df['SK_ID_CURR'].map(avg_days_overdue).fillna(0)

df[df['SK_ID_CURR'] == 100002]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,TOTAL_ACTIVE,TOTAL_CLOSED,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE
675684,100002,6158904,Closed,currency 1,-1125,0,-1038.0,-1038.0,NaN,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
675685,100002,6158905,Closed,currency 1,-476,0,NaN,-48.0,NaN,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
675686,100002,6158906,Closed,currency 1,-1437,0,-1072.0,-1185.0,0.000,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
675687,100002,6158907,Closed,currency 1,-1121,0,-911.0,-911.0,3321.000,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
675688,100002,6158908,Closed,currency 1,-645,0,85.0,-36.0,5043.645,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
675689,100002,6158909,Active,currency 1,-103,0,NaN,NaN,40.500,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
1337779,100002,6158903,Active,currency 1,-1042,0,780.0,NaN,NaN,0,...,2,6,0,0,8,0,-103,-1437,0,0.0
1486113,100002,6113835,Closed,currency 1,-1043,0,62.0,-967.0,0.000,0,...,2,6,0,0,8,0,-103,-1437,0,0.0


In [10]:
g = df.groupby('SK_ID_CURR')

avg_days_credit_update = g['DAYS_CREDIT_UPDATE'].mean()
df['AVERAGE_DAYS_CREDIT_UPDATE'] = df['SK_ID_CURR'].map(avg_days_credit_update)

# AMT_ANNUITY 
# - TOTAL_ANNUITY → sum of active loan's annuities → total monthly obligations

total_annuity = (
    df[df['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR')['AMT_ANNUITY'].sum()
)
df['TOTAL_ANNUITY'] = df['SK_ID_CURR'].map(total_annuity).fillna(0)

df[df['SK_ID_CURR'] == 215354][['SK_ID_CURR', 'AVERAGE_DAYS_CREDIT_UPDATE', 'TOTAL_ANNUITY']]

,SK_ID_CURR,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,215354,-367.272727,0.0
1,215354,-367.272727,0.0
2,215354,-367.272727,0.0
3,215354,-367.272727,0.0
4,215354,-367.272727,0.0
5,215354,-367.272727,0.0
6,215354,-367.272727,0.0
225157,215354,-367.272727,0.0
225158,215354,-367.272727,0.0
225159,215354,-367.272727,0.0


In [11]:
# Make a copy of the original dataframe
df_flat = df.copy()

In [12]:
df_flat = df_flat.drop(columns=['CREDIT_ACTIVE'])
df_flat = df_flat.drop(columns=['CREDIT_CURRENCY'])
df_flat = df_flat.drop(columns=['DAYS_CREDIT'])
df_flat = df_flat.drop(columns=['CREDIT_DAY_OVERDUE'])
df_flat = df_flat.drop(columns=['DAYS_CREDIT_ENDDATE'])
df_flat = df_flat.drop(columns=['DAYS_ENDDATE_FACT'])
df_flat = df_flat.drop(columns=['LOAN_DURATION_SCHEDULED'])
df_flat = df_flat.drop(columns=['LOAN_DURATION_ACTUAL'])
df_flat = df_flat.drop(columns=['LOAN_DEVIATION'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_MAX_OVERDUE'])
df_flat = df_flat.drop(columns=['CNT_CREDIT_PROLONG'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM_DEBT'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM_LIMIT'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM_OVERDUE'])
df_flat = df_flat.drop(columns=['CREDIT_TYPE'])
df_flat = df_flat.drop(columns=['DAYS_CREDIT_UPDATE'])
df_flat = df_flat.drop(columns=['AMT_ANNUITY'])



In [13]:
df_flat = df_flat.drop(columns=['SK_ID_BUREAU'])

In [14]:
cleaned_application_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
22,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,5.0
56,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
61,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
93,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
103,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307412,456145,0,Cash loans,F,N,N,0,162000.0,900000.0,29034.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
307431,456164,0,Cash loans,F,N,N,1,112500.0,334152.0,18256.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307480,456220,0,Cash loans,F,N,N,1,81000.0,1350000.0,39474.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
307484,456228,0,Cash loans,F,Y,N,0,540000.0,545040.0,35617.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df_flat = df_flat.drop_duplicates(subset='SK_ID_CURR')

In [16]:
# # Assuming your application data is called app_df
# merged_df = cleaned_application_df.merge(df_flat, on='SK_ID_CURR', how='left')

# Merge
merged_df = cleaned_application_df.merge(df_flat, on='SK_ID_CURR', how='left')

# Identify rows where SK_ID_CURR does NOT exist in bureau data
no_history_mask = ~merged_df['SK_ID_CURR'].isin(df_flat['SK_ID_CURR'])

# Get bureau columns
bureau_cols = df_flat.columns.tolist()
bureau_cols.remove('SK_ID_CURR')

# Add a binary flag column: 1 = has bureau history, 0 = no bureau history
merged_df['HAS_BUREAU_HISTORY'] = (~no_history_mask).astype(int)

merged_df['HAS_BUREAU_HISTORY'] = (~no_history_mask).map({True: 'Yes', False: 'No'})

In [17]:
# Check duplicates in the application data itself
merged_df['SK_ID_CURR'].duplicated().sum()

np.int64(0)

In [18]:
merged_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY,HAS_BUREAU_HISTORY
0,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,0.0,3.0,0.0,-148.0,-1644.0,0.0,0.0,-314.666667,0.0,Yes
1,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,0.0,2.0,0.0,-339.0,-509.0,0.0,-141.0,-96.500000,4653.0,Yes
2,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0.0,2.0,0.0,-494.0,-630.0,0.0,0.0,-214.500000,0.0,Yes
3,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,0.0,12.0,0.0,-270.0,-2257.0,0.0,0.0,-474.916667,0.0,Yes
4,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,0.0,3.0,0.0,-281.0,-545.0,0.0,0.0,-32.666667,0.0,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,456145,0,Cash loans,F,N,N,0,162000.0,900000.0,29034.0,...,0.0,6.0,0.0,-47.0,-1333.0,1.0,0.0,-475.500000,0.0,Yes
20100,456164,0,Cash loans,F,N,N,1,112500.0,334152.0,18256.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
20101,456220,0,Cash loans,F,N,N,1,81000.0,1350000.0,39474.0,...,0.0,5.0,0.0,-406.0,-1594.0,0.0,0.0,-458.800000,0.0,Yes
20102,456228,0,Cash loans,F,Y,N,0,540000.0,545040.0,35617.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No


In [19]:
merged_df.to_csv('clean_data/final_merged_data.csv', index=False)